In [3]:

import pandas as pd

df = pd.read_csv("../data/processed/telco_customer_churn_clean.csv")

print("Dataset shape:", df.shape)

df.head()

from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)


Dataset shape: (7043, 21)
Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [5]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)



In [6]:

logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained")


Logistic Regression model trained


In [7]:

y_pred = logistic_model.predict(X_test)
y_proba = logistic_model.predict_proba(X_test)[:, 1]

print("First 10 actual values:")
print(y_test.head(10).values)

print("\nFirst 10 predicted values:")
print(y_pred[:10])


First 10 actual values:
[0 0 0 0 0 0 0 0 0 1]

First 10 predicted values:
[0 1 0 0 0 1 0 0 0 0]


In [8]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.8055
Precision: 0.6572
Recall: 0.5588
F1-score: 0.604
ROC-AUC: 0.8421

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


Confusion Matrix:
[[926 109]
 [165 209]]


In [4]:

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd

df = pd.read_csv("../data/processed/telco_customer_churn_clean.csv")

print("Dataset shape:", df.shape)

df.head()

from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

# Build a fresh Logistic Regression pipeline
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]
)

# Parameters to test
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": [None, "balanced"]
}

# Five-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Select settings based on mean F1-score
grid_search_lr = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_search_lr.fit(X_train, y_train)

print("Best parameters:", grid_search_lr.best_params_)
print("Best mean CV F1-score:", round(grid_search_lr.best_score_, 4))


Dataset shape: (7043, 21)
Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Best parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}
Best mean CV F1-score: 0.6295


In [5]:

import pandas as pd

results = pd.DataFrame(grid_search_lr.cv_results_)

results[
    [
        "param_classifier__C",
        "param_classifier__class_weight",
        "mean_test_score",
        "std_test_score",
        "mean_train_score"
    ]
].sort_values(
    by="mean_test_score",
    ascending=False
).round(4)


,param_classifier__C,param_classifier__class_weight,mean_test_score,std_test_score,mean_train_score
3,0.10,balanced,0.6295,0.0209,0.6340
5,1.00,balanced,0.6283,0.0232,0.6336
1,0.01,balanced,0.6279,0.0192,0.6324
7,10.00,balanced,0.6274,0.0239,0.6326
9,100.00,balanced,0.6271,0.0232,0.6314
6,10.00,None,0.5975,0.0302,0.6026
8,100.00,None,0.5972,0.0288,0.6032
4,1.00,None,0.5923,0.0299,0.6023
2,0.10,None,0.5889,0.0283,0.5997
0,0.01,None,0.5812,0.0269,0.5848


In [6]:

# Get the best model selected during tuning
best_logistic_model = grid_search_lr.best_estimator_

# Predict on the test set
tuned_pred = best_logistic_model.predict(X_test)
tuned_proba = best_logistic_model.predict_proba(X_test)[:, 1]

# Evaluate the tuned model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", round(accuracy_score(y_test, tuned_pred), 4))
print("Precision:", round(precision_score(y_test, tuned_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, tuned_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, tuned_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, tuned_proba), 4))

print("\nClassification Report:")
print(classification_report(y_test, tuned_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, tuned_pred))


Accuracy: 0.7417
Precision: 0.5087
Recall: 0.7861
F1-score: 0.6176
ROC-AUC: 0.841

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.73      0.80      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409


Confusion Matrix:
[[751 284]
 [ 80 294]]
